# Descubrimiento rápido del arreglo de loops (WSI a ciegas, versión rápida), Kaggle T4 x2

El #13 encontró solo un arreglo de loops en el Qwen3.5-0.8B (vía MoI empató con p50c), pero tardó **9.1 h**. WSI en sí tardó ~30 min; el resto fue calificar 60 configuraciones. Esta versión quita las tres causas:

1. **Corte de loops arreglado.** vLLM 0.34 cambió la API y el corte se apagaba solo; cada loop corría hasta 16k tokens. Ahora se corta en cuanto se detecta (la calificación corta ahí de todos modos, así que los puntajes no cambian).
2. **vLLM se carga una vez por GPU.** Los pesos editados se escriben en memoria entre configuraciones (con checksum exacto contra el checkpoint horneado).
3. **12 configuraciones en vez de 60** (1–2 capas × 16/todas las neuronas × α 4/8/12), las 3 mejores a la ronda 2, repartidas en las 2 GPUs.

WSI sigue decidiendo solo qué capas, qué neuronas y qué dirección. El examen final (200 GSM8K de test a 16k) usa checkpoints horneados en vLLM nuevo, igual que el #13, con control de dirección al azar y MoI.

**Modelo:** `MODEL` abajo (por defecto **Qwen3.5-4B**). Si el modelo casi no hace loops en GSM8K, se detiene después del diagnóstico y lo dice (eso también es un resultado).

**Uso:** Settings → **Accelerator: GPU T4 x2**, **Internet: On** → **Save Version → Save & Run All (Commit)**. Resultado en el **Output**: `results_discover_fast_<modelo>.json`.

In [ ]:
MODEL = "Qwen/Qwen3.5-4B"          # o "Qwen/Qwen3.5-0.8B" para comparar con el #13
import os, json, subprocess, sys, time, shutil, glob
REPO = "/kaggle/working/wsi"
BASE = "/tmp/base"
W = "/tmp/work"
OUTD = "/kaggle/working"
TAG = MODEL.split("/")[-1].lower().replace(".", "")
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q vllm
!pip install -q flash-linear-attention || echo "fla opcional"
import vllm, transformers, torch; print("vllm", vllm.__version__, "| transformers", transformers.__version__, "| GPUs", torch.cuda.device_count())

In [ ]:
# vLLM compila al arrancar una pieza de FlashInfer que enlaza con -lcuda; Kaggle no trae libcuda.so donde se busca.
real = [l.split("=>")[-1].strip() for l in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout.splitlines()
        if "libcuda.so" in l and "=>" in l]
real += sorted(glob.glob("/usr/local/nvidia/lib64/libcuda.so*")) + sorted(glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*"))
os.makedirs("/tmp/cudalib", exist_ok=True)
if real and not os.path.exists("/tmp/cudalib/libcuda.so"):
    os.symlink(real[0], "/tmp/cudalib/libcuda.so")
os.environ["LIBRARY_PATH"] = "/tmp/cudalib" + (":" + os.environ["LIBRARY_PATH"] if os.environ.get("LIBRARY_PATH") else "")
print("libcuda:", os.path.realpath("/tmp/cudalib/libcuda.so") if os.path.exists("/tmp/cudalib/libcuda.so") else "NO ENCONTRADO")

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git pull -q
S = f"{REPO}/scripts/discover_fast.py"
assert os.path.exists(S), "discover_fast.py no está en GitHub todavía: falta hacer push"
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)

def save(extra=None):
    # resultado parcial o final al Output (tmp + os.replace)
    res = json.load(open(f"{W}/results_discover_fast.json")) if os.path.exists(f"{W}/results_discover_fast.json") else {}
    for f in ("search.json", "timing_fast.json", "frozen.json"):
        if os.path.exists(f"{W}/{f}"):
            res.setdefault("_files", {})[f] = json.load(open(f"{W}/{f}"))
    if os.path.exists(f"{W}/diagnosis_blind.json"):
        d = json.load(open(f"{W}/diagnosis_blind.json")); d.pop("neuron_rank", None)
        res.setdefault("_files", {})["diagnosis"] = d
    res["_final"] = {"commit": COMMIT, "model": MODEL, "notebook": "discover_fast_kaggle.ipynb", **(extra or {})}
    dst = f"{OUTD}/results_discover_fast_{TAG}.json"
    with open(dst + ".tmp", "w") as f:
        json.dump(res, f, indent=1)
    os.replace(dst + ".tmp", dst)
    print("guardado", dst, res["_final"])

In [ ]:
# 1. problemas (GSM8K train para descubrir, test solo para el examen) y respuestas del modelo base
if not os.path.exists(f"{W}/config.json"):
    !python {S} prepare --work {W} --base-dir {BASE} --model-repo {MODEL}
!python -u {S} elicit --work {W}

In [ ]:
# 2. WSI: rankea TODAS las capas por separación loop/normal (d' en datos apartados) y ordena las neuronas.
# En T4 corre en fp32 repartido en las 2 GPUs (fp16 se puede desbordar). Si el modelo casi no hace loops, se detiene aquí.
r = subprocess.run([sys.executable, "-u", S, "diagnose", "--work", W], capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
LOOPS_OK = r.returncode == 0
if not LOOPS_OK:
    save({"complete": True, "status": "stopped_after_diagnose", "reason": (r.stderr.strip().splitlines() or ["?"])[-1]})

In [ ]:
# 3. búsqueda: 12 configuraciones en 32 problemas, las 3 mejores en 64 más; se congela el método antes del examen
if LOOPS_OK:
    !python -u {S} search --work {W}
    !python {S} freeze --work {W} --git-commit {COMMIT}
    print(open(f"{W}/frozen.json").read())
    save({"complete": False, "status": "frozen"})

In [ ]:
# 4. examen: 200 GSM8K test a 16k; base, arreglo siempre prendido, control al azar, y los dos vía MoI
if LOOPS_OK:
    !python -u {S} test --work {W} --ck-root /tmp
    !python {S} report --work {W}
    save({"complete": True, "status": "done"})
for f in glob.glob(f"{W}/gen_gpu*.log"):
    shutil.copy2(f, f"{OUTD}/{TAG}_{os.path.basename(f)}")
shutil.rmtree(REPO, ignore_errors=True)    # el Output solo lleva resultados y logs
print(sorted(os.listdir(OUTD)))